# S23DR 2026 — 3-D Data Visualisation

Interactive Plotly charts for every signal in the dataset:

1. Setup
2. Load a sample
3. Point cloud — coloured by **height (Z)**
4. Point cloud — coloured by **semantic class** (`class_id`)
5. Point cloud — coloured by **view-agreement** (`vote_frac`)
6. Point cloud — coloured by **view count** (`n_views_voted`)
7. Point cloud — coloured by **mask**
8. GT wireframe overlay
9. All signals in one figure
10. Multi-sample gallery
11. Dataset statistics
12. Top-down (bird’s-eye) view
13. Point cloud coloured by `source` mask

## 1 · Setup

In [ ]:
!pip install -q datasets huggingface_hub plotly numpy

In [ ]:
import os
REPO_DIR = "/content/3d_building_construction"
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull origin main
else:
    !git clone https://github.com/12turtleships/3d_building_construction {REPO_DIR}
os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

## 2 · Load samples

Streams a few rows from the public validation split — no local download needed.

In [ ]:
HF_DATASET  = "usm3d/s23dr-2026-sampled_4096_v2"
SPLIT       = "validation"
N_LOAD      = 20
MAX_PTS     = 4096


In [ ]:
import io, zipfile
import numpy as np
from datasets import load_dataset

def unpack(row):
    out = {}
    with zipfile.ZipFile(io.BytesIO(row["data"])) as zf:
        for name in zf.namelist():
            if name.endswith(".npy"):
                out[name[:-4]] = np.load(io.BytesIO(zf.read(name)), allow_pickle=False)
    out["order_id"] = row.get("order_id", "")
    return out

raw = list(load_dataset(HF_DATASET, split=SPLIT, streaming=False).select(range(N_LOAD)))
samples = [unpack(r) for r in raw]
print(f"Loaded {len(samples)} samples")

## 3 · Point cloud — coloured by height (Z)

In [ ]:
import plotly.graph_objects as go
SAMPLE_IDX = 0
s = samples[SAMPLE_IDX]
xyz = s["xyz_norm"]
rng = np.random.default_rng(42)
idx = rng.choice(len(xyz), min(MAX_PTS, len(xyz)), replace=False)
pc  = xyz[idx]
fig = go.Figure(go.Scatter3d(x=pc[:,0], y=pc[:,1], z=pc[:,2], mode="markers",
    marker=dict(size=2.5, color=pc[:,2], colorscale="Viridis",
                colorbar=dict(title="Z (norm)", thickness=12), opacity=0.8)))
fig.update_layout(title=f"{s['order_id']}  —  coloured by height (Z)",
    scene=dict(aspectmode="data"), margin=dict(l=0,r=0,t=40,b=0), height=600)
fig.show()
print(f"{len(pc):,} points rendered")

## 4 · Point cloud — coloured by semantic class (`class_id`)

In [ ]:
s   = samples[SAMPLE_IDX]
xyz = s["xyz_norm"]
cid = s["class_id"]
idx = rng.choice(len(xyz), min(MAX_PTS, len(xyz)), replace=False)
pc  = xyz[idx]; cc = cid[idx]
unique_classes = np.unique(cc)
class_to_idx = {c: i for i, c in enumerate(unique_classes)}
colour_val   = np.array([class_to_idx[c] for c in cc], dtype=float)
fig = go.Figure(go.Scatter3d(x=pc[:,0], y=pc[:,1], z=pc[:,2], mode="markers",
    marker=dict(size=2.5, color=colour_val, colorscale="Rainbow",
                colorbar=dict(title="class_id", thickness=12), opacity=0.85)))
fig.update_layout(title=f"{s['order_id']}  —  coloured by class_id",
    scene=dict(aspectmode="data"), margin=dict(l=0,r=0,t=40,b=0), height=600)
fig.show()

## 5 · Point cloud — coloured by view-agreement (`vote_frac`)

In [ ]:
s   = samples[SAMPLE_IDX]
xyz = s["xyz_norm"]; vf = s["vote_frac"]
idx = rng.choice(len(xyz), min(MAX_PTS, len(xyz)), replace=False)
pc  = xyz[idx]; vc = vf[idx]
fig = go.Figure(go.Scatter3d(x=pc[:,0], y=pc[:,1], z=pc[:,2], mode="markers",
    marker=dict(size=2.5, color=vc, cmin=0.0, cmax=1.0, colorscale="Plasma",
                colorbar=dict(title="vote_frac", thickness=12), opacity=0.85)))
fig.update_layout(title=f"{s['order_id']}  —  coloured by vote_frac",
    scene=dict(aspectmode="data"), margin=dict(l=0,r=0,t=40,b=0), height=600)
fig.show()

## 6 · Point cloud — coloured by view count (`n_views_voted`)

In [ ]:
s   = samples[SAMPLE_IDX]
xyz = s["xyz_norm"]; nv = s["n_views_voted"].astype(float)
idx = rng.choice(len(xyz), min(MAX_PTS, len(xyz)), replace=False)
pc  = xyz[idx]; nc = nv[idx]
fig = go.Figure(go.Scatter3d(x=pc[:,0], y=pc[:,1], z=pc[:,2], mode="markers",
    marker=dict(size=2.5, color=nc, colorscale="Turbo",
                colorbar=dict(title="n_views", thickness=12), opacity=0.85)))
fig.update_layout(title=f"{s['order_id']}  —  coloured by n_views_voted",
    scene=dict(aspectmode="data"), margin=dict(l=0,r=0,t=40,b=0), height=600)
fig.show()

## 7 · Point cloud — coloured by mask

In [ ]:
s    = samples[SAMPLE_IDX]
xyz  = s["xyz_norm"]; mask = s["mask"].astype(bool)
idx  = rng.choice(len(xyz), min(MAX_PTS, len(xyz)), replace=False)
pc   = xyz[idx]; mk = mask[idx]
fig = go.Figure([
    go.Scatter3d(x=pc[mk,0],  y=pc[mk,1],  z=pc[mk,2],  mode="markers",
                 marker=dict(size=2.5, color="royalblue", opacity=0.7),
                 name=f"valid  ({mk.sum():,})"),
    go.Scatter3d(x=pc[~mk,0], y=pc[~mk,1], z=pc[~mk,2], mode="markers",
                 marker=dict(size=2.5, color="tomato",    opacity=0.9),
                 name=f"masked ({(~mk).sum():,})"),
])
fig.update_layout(title=f"{s['order_id']}  —  mask",
    scene=dict(aspectmode="data"), margin=dict(l=0,r=0,t=40,b=0), height=600)
fig.show()

## 8 · GT wireframe overlay

In [ ]:
s    = samples[SAMPLE_IDX]
xyz  = s["xyz_norm"]; segs = s["gt_segments"]
idx  = rng.choice(len(xyz), min(MAX_PTS, len(xyz)), replace=False)
pc   = xyz[idx]
xs, ys, zs = [], [], []
for seg in segs:
    xs += [float(seg[0,0]), float(seg[1,0]), None]
    ys += [float(seg[0,1]), float(seg[1,1]), None]
    zs += [float(seg[0,2]), float(seg[1,2]), None]
fig = go.Figure([
    go.Scatter3d(x=pc[:,0], y=pc[:,1], z=pc[:,2], mode="markers",
                 marker=dict(size=1.5, color="royalblue", opacity=0.3), name="point cloud"),
    go.Scatter3d(x=xs, y=ys, z=zs, mode="lines",
                 line=dict(color="limegreen", width=4), name=f"GT ({len(segs)} segs)"),
])
fig.update_layout(title=f"{s['order_id']}  —  point cloud + GT wireframe",
    scene=dict(aspectmode="data"), margin=dict(l=0,r=0,t=40,b=0), height=650)
fig.show()

## 9 · All signals in one figure

In [ ]:
from plotly.subplots import make_subplots
s    = samples[SAMPLE_IDX]
xyz  = s["xyz_norm"]; segs = s["gt_segments"]
idx  = rng.choice(len(xyz), min(MAX_PTS, len(xyz)), replace=False)
pc   = xyz[idx]
cid  = s["class_id"][idx].astype(float)
vf   = s["vote_frac"][idx]

# showscale=False avoids overlapping colorbars in subplots; subplot titles label the mapping
def _pc_trace(color, cscale, **kwargs):
    return go.Scatter3d(x=pc[:,0], y=pc[:,1], z=pc[:,2], mode="markers",
        marker=dict(size=2.5, color=color, colorscale=cscale, showscale=False, opacity=0.8), **kwargs)

def _seg_trace():
    xs, ys, zs = [], [], []
    for seg in segs:
        xs += [float(seg[0,0]), float(seg[1,0]), None]
        ys += [float(seg[0,1]), float(seg[1,1]), None]
        zs += [float(seg[0,2]), float(seg[1,2]), None]
    return go.Scatter3d(x=xs, y=ys, z=zs, mode="lines", line=dict(color="limegreen", width=4))

fig = make_subplots(rows=2, cols=2,
    specs=[[{"type":"scatter3d"},{"type":"scatter3d"}],[{"type":"scatter3d"},{"type":"scatter3d"}]],
    subplot_titles=("Height (Z)  [Viridis]", "class_id  [Rainbow]",
                    "vote_frac  [Plasma, 0→1]", "GT wireframe  [green]"))
fig.add_trace(_pc_trace(pc[:,2], "Viridis"),              row=1, col=1)
fig.add_trace(_pc_trace(cid,     "Rainbow"),              row=1, col=2)
fig.add_trace(_pc_trace(vf,      "Plasma"),               row=2, col=1)
fig.add_trace(_pc_trace(pc[:,2], "Greys",  opacity=0.25), row=2, col=2)
fig.add_trace(_seg_trace(),                               row=2, col=2)
for sc in ["scene","scene2","scene3","scene4"]:
    fig.update_layout(**{sc: dict(aspectmode="data")})
fig.update_layout(title_text=f"{s['order_id']} — all signals",
    height=900, showlegend=False, margin=dict(l=0,r=0,t=60,b=0))
fig.show()
print("Note: dense horizontal streaks are real SfM structure on architectural edges.")

## 10 · Multi-sample gallery

In [ ]:
from plotly.subplots import make_subplots
GALLERY_N = 6; GALLERY_PTS = 1024
cols = min(3, GALLERY_N); rows = (GALLERY_N + cols - 1) // cols
fig = make_subplots(rows=rows, cols=cols,
    specs=[[{"type":"scatter3d"}]*cols for _ in range(rows)],
    subplot_titles=[samples[i]["order_id"] for i in range(GALLERY_N)])
rng2 = np.random.default_rng(0)
for n in range(GALLERY_N):
    s = samples[n]; xyz = s["xyz_norm"]; segs = s["gt_segments"]
    ri = n // cols + 1; ci = n % cols + 1
    sel = rng2.choice(len(xyz), min(GALLERY_PTS, len(xyz)), replace=False)
    pc = xyz[sel]
    fig.add_trace(go.Scatter3d(x=pc[:,0], y=pc[:,1], z=pc[:,2], mode="markers",
        marker=dict(size=1.2, color=pc[:,2], colorscale="Blues", opacity=0.4),
        showlegend=False), row=ri, col=ci)
    xs, ys, zs = [], [], []
    for seg in segs:
        xs += [float(seg[0,0]), float(seg[1,0]), None]
        ys += [float(seg[0,1]), float(seg[1,1]), None]
        zs += [float(seg[0,2]), float(seg[1,2]), None]
    fig.add_trace(go.Scatter3d(x=xs, y=ys, z=zs, mode="lines",
        line=dict(color="limegreen", width=3), showlegend=False), row=ri, col=ci)
fig.update_layout(title_text=f"Gallery: {GALLERY_N} houses",
    height=380*rows, margin=dict(l=0,r=0,t=60,b=0))
fig.show()

## 11 · Dataset statistics

In [ ]:
from plotly.subplots import make_subplots
n_verts, n_segs, all_vf, all_cid = [], [], [], []
for s in samples:
    gv = s.get("gt_vertices"); gs = s.get("gt_segments")
    if gv is not None: n_verts.append(len(gv))
    if gs is not None: n_segs.append(len(gs))
    all_vf.extend(s["vote_frac"].tolist())
    all_cid.extend(s["class_id"].tolist())
all_vf = np.array(all_vf); all_cid = np.array(all_cid)
unique_cids, counts = np.unique(all_cid, return_counts=True)
fig = make_subplots(rows=2, cols=2,
    subplot_titles=("GT vertex count", "GT edge count", "vote_frac", "class_id"))
fig.add_trace(go.Histogram(x=n_verts, nbinsx=20, marker_color="steelblue"),  row=1, col=1)
fig.add_trace(go.Histogram(x=n_segs,  nbinsx=20, marker_color="seagreen"),   row=1, col=2)
fig.add_trace(go.Histogram(x=all_vf,  nbinsx=40, marker_color="darkorange"), row=2, col=1)
fig.add_trace(go.Bar(x=unique_cids.tolist(), y=counts.tolist(),
                     marker_color="mediumpurple"), row=2, col=2)
fig.update_layout(title_text=f"Statistics across {len(samples)} samples",
    height=700, showlegend=False, margin=dict(l=40,r=20,t=60,b=40))
fig.show()

## 12 · Top-down (bird’s-eye) view

In [ ]:
s    = samples[SAMPLE_IDX]
xyz  = s["xyz_norm"]; segs = s["gt_segments"]; vf = s["vote_frac"]
idx  = rng.choice(len(xyz), min(MAX_PTS, len(xyz)), replace=False)
pc   = xyz[idx]; vc = vf[idx]
fig  = go.Figure()
fig.add_trace(go.Scatter(x=pc[:,0], y=pc[:,1], mode="markers",
    marker=dict(size=2, color=vc, colorscale="Plasma",
                colorbar=dict(title="vote_frac", thickness=12), opacity=0.6)))
for seg in segs:
    fig.add_trace(go.Scatter(x=[seg[0,0], seg[1,0]], y=[seg[0,1], seg[1,1]],
        mode="lines", line=dict(color="limegreen", width=2), showlegend=False))
fig.update_layout(title=f"{s['order_id']}  —  top-down view (XY)",
    xaxis_title="X", yaxis_title="Y", yaxis_scaleanchor="x",
    height=550, margin=dict(l=40,r=20,t=40,b=40), showlegend=False)
fig.show()

## 13 · Point cloud coloured by `source` mask

`source == 1` (green) = points belonging to the **target building**.
`source == 0` (grey)  = neighbouring structures / background.

The procedural pipeline filters to `source == 1` before running; this view shows exactly which points it works with.

In [ ]:
s    = samples[SAMPLE_IDX]
xyz  = s["xyz_norm"]; segs = s["gt_segments"]
src_arr = s.get("source")

if src_arr is None:
    print("'source' array not present in this dataset version.")
else:
    is_target = src_arr.astype(bool)
    idx = rng.choice(len(xyz), min(MAX_PTS, len(xyz)), replace=False)
    pc  = xyz[idx]; mk = is_target[idx]
    xs, ys, zs = [], [], []
    for seg in segs:
        xs += [float(seg[0,0]), float(seg[1,0]), None]
        ys += [float(seg[0,1]), float(seg[1,1]), None]
        zs += [float(seg[0,2]), float(seg[1,2]), None]
    fig = go.Figure([
        go.Scatter3d(x=pc[mk,0],  y=pc[mk,1],  z=pc[mk,2],  mode="markers",
                     marker=dict(size=2.5, color="limegreen", opacity=0.7),
                     name=f"target ({mk.sum():,})"),
        go.Scatter3d(x=pc[~mk,0], y=pc[~mk,1], z=pc[~mk,2], mode="markers",
                     marker=dict(size=1.5, color="lightgrey", opacity=0.3),
                     name=f"background ({(~mk).sum():,})"),
        go.Scatter3d(x=xs, y=ys, z=zs, mode="lines",
                     line=dict(color="dodgerblue", width=4),
                     name=f"GT ({len(segs)} segs)"),
    ])
    fig.update_layout(
        title=f"{s['order_id']}  —  source mask  (green=target building)",
        scene=dict(aspectmode="data"),
        margin=dict(l=0,r=0,t=40,b=0), height=620)
    fig.show()
    print(f"Target: {is_target.sum():,} / {len(is_target):,}  ({is_target.mean()*100:.1f}%)")
